# Agrospheric AI — Streamlit UI Notebook (2 of 2)

Run **Notebook 1 (Backend)** first and download `agrospheric_backend_package.zip`.
Upload that zip here when prompted. This notebook does not duplicate any backend
logic -- it only builds and launches the UI against the exact, already-verified
`pipeline.py` from Notebook 1.

This `app.py` was boot-tested headlessly before being included here (health check
passed, zero import errors) against the real backend, not just written and hoped for.


In [ ]:
# 1. Upload the backend package built by Notebook 1
from google.colab import files
import zipfile, os

print("Upload agrospheric_backend_package.zip:")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_dir = "/content/agrospheric_backend"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(extract_dir)

print("Extracted to", extract_dir)
print(os.listdir(extract_dir))


In [ ]:
# 2. Install Streamlit + everything the backend needs
# (Notebook 1 already trained the model and downloaded fonts -- those come with the zip)
!pip install -q streamlit transformers torch torchvision pillow pydantic matplotlib \
    xgboost shap scikit-learn pandas numpy langgraph \
    sentencepiece gTTS groq reportlab arabic-reshaper python-bidi
print("Dependencies installed.")


In [ ]:
# 3. Set your Groq API key (needed for real Pathologist/Climate agent interpretations)
import os
from google.colab import userdata
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("Groq API key loaded.")
except Exception:
    print("Could not load GROQ_API_KEY -- add it via the key icon in the left sidebar "
          "if you want real Pathologist/Climate output instead of fallback text.")


## 4. Sanity check before launching

Confirms every real module the UI depends on imports cleanly, **before** starting
Streamlit -- so if something's wrong, you find out here with a clear Python traceback,
not as a silent crash in the browser.


In [ ]:
import sys
sys.path.insert(0, "/content/agrospheric_backend")
from tabular_analytics.inference import run_analytical_node
from agent_workflow.translation_agent import translation_agent_node
from agent_workflow.audio_agent import audio_agent_node
from vision_pipeline.inference import run_vision_node
from reporting.report_node import report_node
from pipeline import build_graph
print("All real modules imported cleanly -- safe to launch the UI.")


## 5. Write `app.py`

Sidebar: environmental sliders + language dropdown. Main area: live agent log (via
`graph.stream()`, confirmed to work exactly like this against the real backend),
HITL banner, heatmap + SHAP side by side, localized text + audio, PDF download, and a
debug expander showing raw state.


In [ ]:
%%writefile /content/agrospheric_backend/app.py
import os
import sys
sys.path.insert(0, "/content/agrospheric_backend")

import streamlit as st
from PIL import Image

from pipeline import build_graph
from agent_workflow.language_config import language_choices

st.set_page_config(page_title="Agrospheric AI", layout="wide")
st.title("🌾 Agrospheric AI — Precision Phytopathology & Risk")
st.caption("Multimodal, multi-agent crop disease + environmental risk pipeline")

with st.sidebar:
    st.header("Environmental Vector")
    temperature_c = st.slider("Temperature (°C)", 5.0, 45.0, 27.5)
    humidity_pct = st.slider("Humidity (%)", 0.0, 100.0, 88.0)
    rainfall_mm_14d = st.slider("Rainfall, 14 days (mm)", 0.0, 150.0, 95.0)
    soil_n = st.slider("Soil Nitrogen", 0.0, 180.0, 70.0)
    soil_p = st.slider("Soil Phosphorus", 0.0, 90.0, 30.0)
    soil_k = st.slider("Soil Potassium", 0.0, 140.0, 45.0)
    soil_moisture_pct = st.slider("Soil Moisture (%)", 0.0, 100.0, 62.0)

    st.header("Localization")
    target_language = st.selectbox("Target language", language_choices())

uploaded_file = st.file_uploader("Upload a crop leaf photo", type=["jpg", "jpeg", "png"])
run_clicked = st.button("Run Diagnosis", type="primary", use_container_width=True)

if run_clicked:
    if uploaded_file is None:
        st.warning("Please upload a leaf photo first.")
    else:
        image = Image.open(uploaded_file).convert("RGB")
        tabular_input = {
            "temperature_c": temperature_c, "humidity_pct": humidity_pct,
            "rainfall_mm_14d": rainfall_mm_14d, "soil_n": soil_n,
            "soil_p": soil_p, "soil_k": soil_k, "soil_moisture_pct": soil_moisture_pct,
        }

        st.subheader("Live agent log")
        log_area = st.container()
        graph = build_graph()
        final_state = {}

        with st.spinner("Running the multi-agent pipeline..."):
            for step in graph.stream({
                "image": image, "tabular_input": tabular_input, "target_language": target_language,
            }):
                node_name = list(step.keys())[0]
                log_area.write(f"✅ `{node_name}` completed")
                final_state.update(step[node_name])

        if final_state.get("used_fallback"):
            st.warning(
                "⚠️ Some real modules weren't available for this run — showing "
                "demo-safe fallback data instead of crashing."
            )

        vision = final_state.get("vision_output", {})
        analytical = final_state.get("analytical_output", {})

        st.divider()

        if vision.get("is_ambiguous"):
            st.error(
                "🔴 AMBIGUOUS — AGRONOMIST REVIEW REQUIRED. No autonomous treatment "
                "recommendation has been generated for this case."
            )
        else:
            band = analytical.get("risk_band", "Moderate")
            banner = {"Low": st.success, "Moderate": st.warning, "High": st.error}.get(band, st.info)
            banner(f"🟢 Cleared for action plan — Risk band: {band}")

        col1, col2, col3 = st.columns(3)
        col1.metric("Disease", vision.get("disease_class", "—").replace("___", " — ").replace("_", " "))
        col2.metric("Confidence", f"{vision.get('confidence_score', 0) * 100:.1f}%")
        col3.metric("14-day risk", f"{analytical.get('14_day_risk_pct', 0):.1f}%")

        st.divider()
        img_col1, img_col2 = st.columns(2)
        with img_col1:
            st.subheader("Attention heatmap")
            heatmap_path = final_state.get("heatmap_path")
            if heatmap_path and os.path.exists(heatmap_path):
                st.image(heatmap_path, use_container_width=True)
            else:
                st.caption("No heatmap available for this run.")
        with img_col2:
            st.subheader("SHAP stressors")
            if os.path.exists("shap_chart.png"):
                st.image("shap_chart.png", use_container_width=True)
            else:
                st.caption("No SHAP chart available for this run.")

        st.divider()
        st.subheader(f"Localized report ({target_language})")
        st.info(final_state.get("translated_text", "—"))
        audio = final_state.get("audio_output") or {}
        if audio.get("tts_status") == "success" and audio.get("audio_file_path") and os.path.exists(audio["audio_file_path"]):
            st.audio(audio["audio_file_path"])
        elif audio.get("tts_status") == "skipped_unsupported_language":
            st.caption(f"Audio not available for {target_language}: {audio.get('skip_reason', '')}")

        if final_state.get("pdf_path") and os.path.exists(final_state["pdf_path"]):
            with open(final_state["pdf_path"], "rb") as f:
                st.download_button(
                    "📥 Download PDF Report", f, file_name="agrospheric_report.pdf",
                    mime="application/pdf", use_container_width=True,
                )

        with st.expander("Debug: raw pipeline state"):
            debug_view = {k: v for k, v in final_state.items() if k != "vision_output"}
            debug_view["vision_output"] = {k: v for k, v in vision.items() if k != "attention_matrix"}
            st.json(debug_view)


## 6. Launch Streamlit (ONE tunnel, not two)

The earlier notebook accidentally started two competing `localtunnel` processes on
the same port, which is a real cause of link/connection weirdness. This starts
exactly one.


In [ ]:
!npm install -g localtunnel -q


In [ ]:
import subprocess, time, urllib.request

# Kill any stray previous instances first (avoids the double-tunnel problem)
subprocess.run(["pkill", "-f", "streamlit run"], capture_output=True)
subprocess.run(["pkill", "-f", "lt --port"], capture_output=True)
time.sleep(2)

# Start Streamlit in the background
subprocess.Popen([
    "streamlit", "run", "/content/agrospheric_backend/app.py",
    "--server.port", "8501", "--server.headless", "true",
])
time.sleep(6)

# Confirm it actually booted before opening a tunnel to it
try:
    resp = urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=5)
    print("Streamlit health check:", resp.read().decode())
except Exception as e:
    print("Streamlit did NOT start cleanly -- check the cell above for import errors.")
    print("Error:", e)


In [ ]:
import urllib.request
print("Tunnel password (paste this if localtunnel asks for it):")
print(urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip())

# Exactly one tunnel
get_ipython().system_raw("lt --port 8501 > /content/localtunnel_link.txt 2>&1 &")
import time; time.sleep(4)
with open("/content/localtunnel_link.txt") as f:
    print(f.read())


## 7. If something's still wrong

- **Streamlit health check failed above** → scroll up to Section 4's import check; the
  traceback there tells you exactly which real module is broken, not a generic crash.
- **Link loads but a specific feature errors** → open the "Debug: raw pipeline state"
  expander at the bottom of a run; check `used_fallback` -- if `True`, a real module
  silently failed to import inside the running app (re-run Section 4 to see why).
- **Odia audio missing** → expected, not a bug -- gTTS doesn't support Odia; the
  translated text still shows.
- **A specific language's Section 3 looks like boxes** → per the font README, this can
  happen if real NLLB output leaves some words untranslated (mixed script); spot-check
  your actual demo languages once, not all 11.
